# TGI-metric extraction — the Stein/Bruno panel

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

Every simulated trajectory is summarized into the derived metrics oncology pharmacometrics reports (spec §3, §6): depth of response, nadir / time-to-growth, the **tumor growth-rate constant k_g** (the strongly prognostic Stein/Bruno quantity), the shrinkage-rate constant k_s, and the RECIST-style duration of response. The extractor is model-agnostic, so it recovers the *generating* rates of the biexponential kernel and the Claret growth rate alike.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos

ds = onkos.load()
t = np.linspace(0, 156, 313)
tr = onkos.simulate(ds, 'tgi_metrics.wang_2009.biexponential', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.2, t=t)
for k, v in tr.metrics.items():
    print(f'{k:<28} {v:.4f}')

In [ ]:
# The extractor recovers the biexponential kernel's GENERATING rates (kg, ks).
rec = ds['tgi_metrics.wang_2009.biexponential']
kg_gen, ks_gen = rec['kg'].central, rec['ks'].central
m = onkos.simulate(ds, 'tgi_metrics.wang_2009.biexponential', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0, t=t).metrics
print(f"kg: extracted {m['tumor_growth_rate_kg']:.4f} vs generating {kg_gen}")
print(f"ks: extracted {m['tumor_shrinkage_rate_ks']:.4f} vs generating {ks_gen}")
assert abs(m['tumor_growth_rate_kg'] - kg_gen) / kg_gen < 0.20
assert abs(m['tumor_shrinkage_rate_ks'] - ks_gen) / ks_gen < 0.20

In [ ]:
# For the Claret model the extracted growth rate recovers kL; a durable response
# may show no regrowth in-window (k_g = nan), which is reported honestly.
cl = onkos.simulate(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=1.0, t=t).metrics
print('Claret kL (param):', ds['resistance.claret_2009.tgi']['kL'].central)
print('Claret extracted k_g:', round(cl['tumor_growth_rate_kg'], 4))
print('Claret depth of response:', round(cl['depth_of_response'], 3))

plt.semilogy(t, tr.tumor_size)
plt.axhline(tr.tumor_size[0]*0.7, ls='--', color='green'); plt.xlabel('weeks'); plt.ylabel('SLD (log)')
plt.title('biexponential trajectory (V-shape: k_s then k_g)');